# 14.2-5 Check known edges
check how many times edges have been checked for collisions to prove that edges are only checked once.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

# Import custom modules
from planners.SBL import BidirectionalSBL
from modules import randomScene, IPVISsbl
from lecture_examples.IPPerfMonitor import IPPerfMonitor

In [ ]:
SCENE_LIMITS = np.array([[0,22],[0,22]])
STEPS = 50

In [ ]:
time_naive = 0.
time_adaptive = 0.
n_missed_collisions = 0
n_no_path = 0
SHOW_MISSED_COLLISIONS = False

IPPerfMonitor.clearData()
N_SCENES = 200
for i in range(N_SCENES):
    benchmark = randomScene.create_random_benchmark(SCENE_LIMITS)
    planner_naive = BidirectionalSBL(benchmark.collisionChecker,{"count_edge_checks":True,
                                                                 "collision_check": {
                                                                     "adaptive": False,
                                                                     "steps": STEPS,}})
    planner_adaptive = BidirectionalSBL(benchmark.collisionChecker,{"count_edge_checks":True,
                                                                    "collision_check": {
                                                                        "adaptive":True,
                                                                        "steps": STEPS,
                                                                        "epsilon":0.0001}})
    path_naive = planner_naive.planPath(list(benchmark.startList[0]), list(benchmark.goalList[0]))
    time_naive += IPPerfMonitor.get_time("__call__")
    IPPerfMonitor.clearData()
    path_adaptive = planner_adaptive.planPath(list(benchmark.startList[0]), list(benchmark.goalList[0]))
    time_adaptive += IPPerfMonitor.get_time("__call__")
    IPPerfMonitor.clearData()
    edge_check_counts = planner_adaptive.collision_check_counter

    # 14.2 Check that no edge is checked for collisions twice
    assert np.any(np.array(list(edge_check_counts.values()))<2), f"Edge doublechecked for collision {edge_check_counts.values()}"

    if path_adaptive:
        # 14.4 Check that path goes from start to goal point
        assert np.all(path_adaptive[0].coordinates == benchmark.startList[0]), "Start points not matching"
        assert np.all(path_adaptive[-1].coordinates == benchmark.goalList[0]), "Goal points not matching"


        # 14.5 Check that all edges on path are collision free (complete line test as reference)
        colliding_edges = []
        for i_node in range(len(path_adaptive)-2):
            assert not benchmark.collisionChecker.pointInCollision(path_adaptive[i_node].coordinates), f"Colliding node {path_adaptive[i_node]}"
            if benchmark.collisionChecker.lineInCollisionExact(path_adaptive[i_node].coordinates,path_adaptive[i_node+1].coordinates):
                colliding_edges.append(path_adaptive[i_node:i_node+2])
        if colliding_edges:
            n_missed_collisions += 1
            if SHOW_MISSED_COLLISIONS:
                fig = plt.figure(figsize=(10, 10))
                ax = fig.add_subplot(1, 1, 1)
                IPVISsbl.sblVisualize(planner_adaptive,path_adaptive,ax)
                ax.set_title(f"Colliding edges {colliding_edges}")
                plt.show()
    else:
        n_no_path += 1
print(f"Missed collisions: {n_missed_collisions} ({n_missed_collisions*100/N_SCENES}%)")
print(f"No path found: {n_no_path} ({n_no_path*100/N_SCENES}%)")


Most paths do not have any collisions. Sometimes, an edge runs through a tiny section of an obstacle such as a corner or the edge slightly touches the surface. This means that there are only collisions at a very short section of the edge. This is hard to find with an adaptive line test with limited points.

In [ ]:
plt.bar(["naive","adaptive"],[time_naive*1_000/N_SCENES,time_adaptive*1_000/N_SCENES])
plt.title(f"Edge collision check: {np.round((1-time_adaptive/time_naive)*100,2)}% time saving in adaptive mode")
plt.ylabel("Duration per edge / ms")
plt.show()